# Day 14 — From prompt to agent: the Echo agent

**Phase 2 · From Generative to Agentic · ~45 min · Needs a provider**

## 🎯 Objective
Combine the LLM brain (Phase 1) with the loop + tools (this week) into your **first
complete agent**: a model that reads a tool catalog, decides in JSON, and your code
runs the tool — looping until it answers.

## 🧠 The big picture
```mermaid
flowchart TD
    U["🙋 User goal"] --> M["🧠 LLM decides (JSON)"]
    M -->|action: tool| D["🛠️ Dispatch tool"] --> O["👀 Observation"] --> M
    M -->|action: final| A["✅ Final answer"]
```

Each turn the model replies with ONLY JSON:
```json
{"action": "tool", "tool": "calculator", "args": {"expression": "2+2"}}
{"action": "final", "answer": "..."}
```
This is **manual function-calling** — tomorrow's Phase continues with the *native*
version, but building it by hand means no framework is ever a black box.

## 🔬 Exercise
Implement the loop in `run()`: ask the model, parse JSON, dispatch tools, feed the
**observation** back, and stop at `final` or a step budget.

## ✅ Done when
- It uses the right tool(s) and then gives a final answer, without looping forever.

## 📝 Quiz (journal)
1. Difference between a plain LLM call and an agent?
2. Why feed the observation back into `messages`?
3. Why keep temperature low for the decision step?

## 🎉 You built an agent!
## 🔭 Next: Phase 3 — Tools, memory & planning
Native tool calling, ReAct, real memory, an `Agent` class, and your first capstone.


### ▶ Run this first

In [ ]:
# Run me first: make the course's shared/ package importable from this notebook.
import sys, pathlib
for _cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_cand / 'shared' / 'llm.py').exists():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break


## 🔬 Your turn
Complete the `TODO`s below, then run the cell (**Shift+Enter**).

In [ ]:
import json

from shared.llm import chat
from shared.tools import calculator, clock

TOOLS = {"calculator": calculator, "clock": clock}

SYSTEM = """You are a tool-using agent. Each turn reply with ONLY a JSON object.
Choose ONE:
  {"action":"tool","tool":"calculator","args":{"expression":"2+2"}}
  {"action":"tool","tool":"clock","args":{}}
  {"action":"final","answer":"your answer"}
Tools: calculator(expression), clock(). Use tools to get facts, then answer. JSON only."""


def _strip_fences(s):
    s = s.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if "\n" in s:
            s = s.split("\n", 1)[1]
    return s.strip()


def run(goal, max_steps=5):
    """Agent loop.

    TODO:
      messages = [system, user=goal]
      for step in range(max_steps):
        raw = chat(messages, temperature=0); step = json.loads(_strip_fences(raw))
        if action == 'final': return answer
        run the tool, append assistant raw + user 'Observation: ...'
      return 'stopped'
    """
    raise NotImplementedError


if __name__ == "__main__":
    print(run("What is 17 * 23, and what time is it?"))


---
---
## 🔒 Solution
*Try it yourself first!* Run the cell below only when you're ready to compare.

In [1]:
import json

from shared.llm import chat
from shared.tools import calculator, clock

TOOLS = {"calculator": calculator, "clock": clock}

SYSTEM = """You are a tool-using agent. Each turn reply with ONLY a JSON object.
Choose ONE:
  {"action":"tool","tool":"calculator","args":{"expression":"2+2"}}
  {"action":"tool","tool":"clock","args":{}}
  {"action":"final","answer":"your answer"}
Tools: calculator(expression), clock(). Use tools to get facts, then answer. JSON only."""


def _strip_fences(s):
    s = s.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if "\n" in s:
            s = s.split("\n", 1)[1]
    return s.strip()


def run(goal, max_steps=5, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": goal}]
    for step_num in range(1, max_steps + 1):
        raw = chat(messages, temperature=0)
        try:
            step = json.loads(_strip_fences(raw))
        except json.JSONDecodeError:
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": "Reply with ONLY the JSON object."})
            continue
        if step.get("action") == "final":
            return step.get("answer", "(no answer)")
        name = step.get("tool", "")
        tool = TOOLS.get(name)
        obs = tool(**step.get("args", {})) if tool else f"Error: unknown tool '{name}'"
        if verbose:
            print(f"[step {step_num}] {name}({step.get('args', {})}) -> {obs}")
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return "stopped: step budget exceeded"


if __name__ == "__main__":
    print("FINAL:", run("What is 17 * 23, and what time is it?"))


TypeError: clock() got an unexpected keyword argument 'format'